In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
schema = "bronze"
volume_path = f"/Volumes/{catalog}/{schema}/raw_sources"

print("BRONZE LAYER - RAW DATA INGESTION")
print("=" * 80)

# Define files configuration
files_config = {
    "cust_info": ("crm", "cust_info"),
    "prd_info": ("crm", "prd_info"),
    "sales_details": ("crm", "sales_details"),
    "CUST_AZ12": ("erp", "cust_az12"),
    "LOC_A101": ("erp", "loc_a101"),
    "PX_CAT_G1V2": ("erp", "px_cat_g1v2")
}

# Ingest each file
print("\nIngesting files into Bronze layer:")
for file_name, (source, table_name) in files_config.items():
    file_path = f"{volume_path}/{file_name}.csv"
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    
    full_table_name = f"{catalog}.{schema}.{source}_{table_name}"
    df.write.mode("overwrite").format("delta").saveAsTable(full_table_name)
    
    row_count = df.count()
    col_count = len(df.columns)
    print(f"  {full_table_name}: {row_count:,} rows, {col_count} columns")

# Verify all tables
print("\nVerification:")
bronze_tables = spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()
print(f"Total tables created: {len(bronze_tables)}")

for table in bronze_tables:
    table_name = table.tableName
    row_count = spark.sql(f"SELECT COUNT(*) FROM {catalog}.{schema}.{table_name}").collect()[0][0]
    print(f"  {table_name}: {row_count:,} rows")

print("\nBronze layer ingestion completed.")